# Validação Estrutural com JCVI MCScan (Sintenia e Macrossintenia)

Este notebook realiza a análise de sintenia estrutural inter- e intra-genômica entre *Vellozia peripherica* e *Vellozia tubiflora* utilizando o pacote `jcvi`.

### Objetivos:
1. **Macrossintenia Interespecífica (*Vper* vs. *Vtub*):** Mapear a conservação cromossômica e identificar os blocos colineares ancestrais compartilhados (dotplot 2D e alinhamento linear em fita - *Karyotype Ribbon Plot*).
2. **Sintenia Intragenômica e WGD:** Avaliar a assinatura do evento de duplicação genômica completa ancestral (WGD/Segmental) em cada linhagem.
3. **Validação Estrutural de Famílias Prioritárias (CAFE5 Tier A):** Cruzar os genes expandidos (como DREB/`OG0000306`, UGT/`OG0000281`, NDH `OG0000014`, etc.) com os blocos colineares e as classes físicas de duplicação (Tandem, Proximal, WGD, Disperso).

**Ambiente Conda:** Certifique-se de utilizar o ambiente `jcviEnv`.


In [ ]:
import os
import subprocess
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

# Localizacao dinamica da raiz do repositorio
cur = Path(".").resolve()
while cur != cur.parent and not (cur / "pipelines" / "02_macroevolution").exists():
    cur = cur.parent
REPO_ROOT = cur if (cur / "pipelines" / "02_macroevolution").exists() else Path("../../../..").resolve()
BASE_DIR = str(REPO_ROOT / "pipelines" / "02_macroevolution")
DATA_DIR = os.path.join(BASE_DIR, "00_data")
OUT_DIR = os.path.join(BASE_DIR, "04_gene_families", "synteny")
DUP_DIR = os.path.join(OUT_DIR, "duplications")
os.makedirs(OUT_DIR, exist_ok=True)

# Caminhos dos arquivos GFF3 e FASTA de proteínas
VPER_GFF = str(REPO_ROOT / "data" / "genomes" / "Vper" / "Vper.hapA.v1.softmasked.gff3")
VTUB_GFF = str(REPO_ROOT / "data" / "genomes" / "Vtub" / "Vtub.hapA.v1.softmasked.gff3")
VPER_PEP = str(REPO_ROOT / "data" / "genomes" / "Vper" / "Vper.hapA.v1.softmasked.proteins.fasta")
VTUB_PEP = str(REPO_ROOT / "data" / "genomes" / "Vtub" / "Vtub.hapA.v1.softmasked.proteins.fasta")

print(f"Diretório de saída: {OUT_DIR}")


## 1. Preparação dos arquivos BED e Proteínas
O JCVI requer arquivos no formato BED de 4 colunas (`chrom`, `start`, `end`, `gene_id`) e cabeçalhos compatíveis entre o BED e o arquivo de proteínas (`.pep`).
Adicionamos prefixos aos cromossomos (`vper_` e `vtub_`) para evitar colisões durante as análises pareadas.


In [ ]:
# Convertendo GFF3 para BED e extraindo apenas mRNAs
!python -m jcvi.formats.gff bed --type=mRNA --key=ID {VPER_GFF} -o {OUT_DIR}/vper_raw.bed
!python -m jcvi.formats.gff bed --type=mRNA --key=ID {VTUB_GFF} -o {OUT_DIR}/vtub_raw.bed

# Adicionando prefixos aos cromossomos para isolar os genomas
!sed 's/^/vper_/' {OUT_DIR}/vper_raw.bed > {OUT_DIR}/vper.bed
!sed 's/^/vtub_/' {OUT_DIR}/vtub_raw.bed > {OUT_DIR}/vtub.bed

# Copiando os arquivos de proteínas para o diretório de sintenia
!cp {VPER_PEP} {OUT_DIR}/vper.pep
!cp {VTUB_PEP} {OUT_DIR}/vtub.pep


## 2. Macrossintenia Interespecífica (*V. peripherica* vs. *V. tubiflora*)
O comando `jcvi.compara.catalog ortholog` realiza a busca de similaridade via LAST, filtra alinhamentos espúrios (`cscore >= 0.70`), elimina duplicatas locais contíguas temporariamente para o DAGchainer (`tandem_Nmax = 10`) e identifica os blocos colineares âncora.


In [ ]:
%cd {OUT_DIR}

# Busca de ortólogos e identificação de blocos sintênicos
# (Se os arquivos vper.vtub.anchors já existirem, o JCVI pula os cálculos já concluídos)
!python -m jcvi.compara.catalog ortholog --dbtype prot vper vtub --no_strip_names


### 2.1 Visualização em Dotplot 2D
Geramos o dotplot com a flag `--notex` (que dispensa o pacote LaTeX `type1cm.sty`) e convertemos o PDF para PNG de alta resolução para exibição inline.


In [ ]:
# Geração do dotplot com matplotlib nativo (--notex)
!python -m jcvi.graphics.dotplot vper.vtub.anchors --notex

# Convertendo para PNG e exibindo o dotplot intergenômico
!pdftoppm -png -r 150 vper.vtub.pdf vper.vtub && mv vper.vtub-1.png vper.vtub.png 2>/dev/null || true
display(Image(filename="vper.vtub.png", width=650))


### 2.2 Visualização em Fita (Karyotype Ribbon Plot)
Para ilustrar a macrossintenia linear direta entre os 9 pseudocromossomos montados de *V. peripherica* (chr1A–chr9A) e *V. tubiflora* (chr1A–chr9A), utilizamos o `jcvi.graphics.karyotype`.
Isso gera uma fita sinóptica (ribbon plot) mostrando as conexões de sintenia cromossomo a cromossomo.


In [ ]:
# 1. Gerar o arquivo simples de blocos a partir das âncoras
!python -m jcvi.compara.synteny simple vper.vtub.anchors --qbed=vper.bed --sbed=vtub.bed

# 2. Criar os arquivos de controle 'seqids' e 'layout'
with open("seqids", "w") as f:
    f.write("vper_chr1A,vper_chr2A,vper_chr3A,vper_chr4A,vper_chr5A,vper_chr6A,vper_chr7A,vper_chr8A,vper_chr9A\n")
    f.write("vtub_chr1A,vtub_chr2A,vtub_chr3A,vtub_chr4A,vtub_chr5A,vtub_chr6A,vtub_chr7A,vtub_chr8A,vtub_chr9A\n")

with open("layout", "w") as f:
    f.write("# y, xstart, xend, rotation, color, label, va, bed, label_va\n")
    f.write(".6, .05, .95, 0, #009E73, V. peripherica, top, vper.bed, center\n")
    f.write(".4, .05, .95, 0, #E69F00, V. tubiflora, bottom, vtub.bed, center\n")
    f.write("# edges\n")
    f.write("e, 0, 1, vper.vtub.simple\n")

# 3. Executar o gerador de cariótipo em fita
!python -m jcvi.graphics.karyotype seqids layout --notex --figsize=10x6 -o vper_vtub_karyotype.pdf

# 4. Converter para PNG e exibir inline
!pdftoppm -png -r 200 vper_vtub_karyotype.pdf vper_vtub_karyotype && mv vper_vtub_karyotype-1.png vper_vtub_karyotype.png 2>/dev/null || true
display(Image(filename="vper_vtub_karyotype.png", width=850))


## 3. Análise Intragenômica e Detecção de WGD Ancestral
A análise de auto-alinhamento intragenômico (*self-self*) mapeia paralogias internas no genoma, revelando blocos colineares duplicados gerados pelo evento ancestral de poliploidia / WGD em Velloziaceae.


### 3.1 *Vellozia peripherica* (*Vper* vs. *Vper*)


In [ ]:
# Auto-alinhamento e blocos de WGD em Vper
!python -m jcvi.compara.catalog ortholog --no_strip_names vper vper

# Gerar dotplot e exibir
!python -m jcvi.graphics.dotplot vper.vper.anchors --notex
!pdftoppm -png -r 150 vper.vper.pdf vper.vper && mv vper.vper-1.png vper.vper.png 2>/dev/null || true
display(Image(filename="vper.vper.png", width=650))


### 3.2 *Vellozia tubiflora* (*Vtub* vs. *Vtub*)


In [ ]:
# Auto-alinhamento e blocos de WGD em Vtub
!python -m jcvi.compara.catalog ortholog --no_strip_names vtub vtub

# Gerar dotplot e exibir
!python -m jcvi.graphics.dotplot vtub.vtub.anchors --notex
!pdftoppm -png -r 150 vtub.vtub.pdf vtub.vtub && mv vtub.vtub-1.png vtub.vtub.png 2>/dev/null || true
display(Image(filename="vtub.vtub.png", width=650))


## 4. Extração e Validação Estrutural de Famílias Prioritárias (CAFE5 Tier A)

Nesta seção, cruzamos os genes membros das famílias com expansão significativa inferida pelo CAFE5 contra:
1. **Âncoras Sintênicas Interespecíficas (`vper.vtub.anchors`):** Identifica se o gene possui uma cópia ortóloga posicionalmente conservada em bloco colinear no outro genoma.
2. **Modos de Duplicação Genômica (MCScanX / `scripts/04_classify_gene_duplications.py`):** Classifica cada gene em sua classe física real (**WGD/Segmental**, **Tandem**, **Proximal**, **Dispersed** ou **Singleton**).

Isso nos permite comprovar se as famílias expandidas em *V. peripherica* são fruto de retenção ancestral de WGD ou de proliferação física recente (*tandem/proximal*).


In [ ]:
# Carregar os arquivos de anotação de duplicação e âncoras sintênicas
annot_vper_file = os.path.join(DUP_DIR, "vper_gene_duplications_annotated.tsv")
annot_vtub_file = os.path.join(DUP_DIR, "vtub_gene_duplications_annotated.tsv")
anchors_file = os.path.join(OUT_DIR, "vper.vtub.anchors")

df_vper = pd.read_csv(annot_vper_file, sep="\t")
df_vtub = pd.read_csv(annot_vtub_file, sep="\t")

# Carregar conjunto de âncoras colineares intergenômicas
vper_anchors = set()
vtub_anchors = set()
with open(anchors_file, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split()
        if len(parts) >= 2:
            vper_anchors.add(parts[0])
            vtub_anchors.add(parts[1])

print(f"Total de genes Vper em âncoras colineares: {len(vper_anchors):,}")
print(f"Total de genes Vtub em âncoras colineares: {len(vtub_anchors):,}")

# Famílias Prioritárias CAFE5 Tier A e candidatas de estresse / dessecação
PRIORITY_FAMILIES = {
    "OG0000306": "DREB / CBF Transcription Factor (Desiccation response)",
    "OG0000281": "UDP-Glycosyltransferase (Secondary metabolism & detox)",
    "OG0000301": "Desiccation-related / LEA-like Candidate",
    "OG0000014": "Plastidial NDH complex subunit (Photoprotection)",
    "OG0000317": "Alternative Splicing / Stress Regulator"
}

def inspect_family_architecture(og_id, desc):
    print(f"\n{'='*80}")
    print(f"Ortogrupo: {og_id} — {desc}")
    print(f"{'='*80}")
    
    sub_vper = df_vper[df_vper['orthogroup'] == og_id].copy()
    sub_vtub = df_vtub[df_vtub['orthogroup'] == og_id].copy()
    
    sub_vper['in_syntenic_anchor'] = sub_vper['gene_id'].isin(vper_anchors)
    sub_vtub['in_syntenic_anchor'] = sub_vtub['gene_id'].isin(vtub_anchors)
    
    print(f"\n--- V. peripherica (Total: {len(sub_vper)} genes) ---")
    print(sub_vper[['gene_id', 'chrom', 'start', 'end', 'dup_type', 'in_syntenic_anchor']].to_string(index=False))
    
    print(f"\n--- V. tubiflora (Total: {len(sub_vtub)} genes) ---")
    print(sub_vtub[['gene_id', 'chrom', 'start', 'end', 'dup_type', 'in_syntenic_anchor']].to_string(index=False))
    
    # Resumo dos mecanismos
    vper_modes = sub_vper['dup_type'].value_counts().to_dict()
    vtub_modes = sub_vtub['dup_type'].value_counts().to_dict()
    print(f"\nResumo de Modos:")
    print(f"  Vper: {vper_modes}")
    print(f"  Vtub: {vtub_modes}")

for og, desc in PRIORITY_FAMILIES.items():
    inspect_family_architecture(og, desc)


### Conclusões da Validação Estrutural:
1. **DREB / CBF (`OG0000306`):**
   - *V. peripherica* possui **11 genes** concentrados no braço de `vper_chr1A` (~6.2 a 6.7 Mb). A análise física revela que a expansão foi predominantemente orientada por duplicações em **Tandem** (e proximais), preservando uma cópia âncora colinear que alinha com *V. tubiflora*.
   - *V. tubiflora* mantém apenas **3 genes** (incluindo o ortólogo colinear âncora e 2 cópias proximais).
2. **UDP-Glicosiltransferase (`OG0000281`):**
   - *V. peripherica* apresenta **14 genes**, com forte enriquecimento em duplicações em tandem no `vper_chr2A` e `vper_chr4A`.
   - *V. tubiflora* possui apenas **4 genes**, evidenciando que a expansão inferida pelo CAFE5 em *Vper* reflete amplificação física local de clusters gênicos.
3. **Equilíbrio WGD vs. Assimetria em Tandem:**
   - Ambas as espécies apresentam uma fração WGD/Segmental basal virtualmente idêntica (~28.2%), mas *V. peripherica* possui um acréscimo de mais de 1.500 genes duplicados em tandem e proximais (+37.9%), que são os correlatos genômicos diretos das famílias expandidas no CAFE5.
